In [ ]:
!pip install apyori
from apyori import apriori
import pandas as pd

df = pd.read_csv("Bakery.csv")
df

,TransactionNo,Items,DateTime,Daypart,DayType
0,1,Bread,2016-10-30 09:58:11,Morning,Weekend
1,2,Scandinavian,2016-10-30 10:05:34,Morning,Weekend
2,2,Scandinavian,2016-10-30 10:05:34,Morning,Weekend
3,3,Hot chocolate,2016-10-30 10:07:57,Morning,Weekend
4,3,Jam,2016-10-30 10:07:57,Morning,Weekend
...,...,...,...,...,...
20502,9682,Coffee,2017-09-04 14:32:58,Afternoon,Weekend
20503,9682,Tea,2017-09-04 14:32:58,Afternoon,Weekend
20504,9683,Coffee,2017-09-04 14:57:06,Afternoon,Weekend
20505,9683,Pastry,2017-09-04 14:57:06,Afternoon,Weekend


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20507 entries, 0 to 20506
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   TransactionNo  20507 non-null  int64 
 1   Items          20507 non-null  object
 2   DateTime       20507 non-null  object
 3   Daypart        20507 non-null  object
 4   DayType        20507 non-null  object
dtypes: int64(1), object(4)
memory usage: 801.2+ KB


In [ ]:
df['DayType'].unique()


array(['Weekend', 'Weekday'], dtype=object)

In [ ]:
df['Daypart'].unique()

array(['Morning', 'Afternoon', 'Evening', 'Night'], dtype=object)

In [ ]:
# Find out information about the types of data in the dataframe
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20507 entries, 0 to 20506
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   TransactionNo  20507 non-null  int64 
 1   Items          20507 non-null  object
 2   DateTime       20507 non-null  object
 3   Daypart        20507 non-null  object
 4   DayType        20507 non-null  object
dtypes: int64(1), object(4)
memory usage: 801.2+ KB


In [ ]:
# Search for missing values in the dataframe
print("Missing values:\n", df.isnull().sum())

Missing values:
 TransactionNo    0
Items            0
DateTime         0
Daypart          0
DayType          0
dtype: int64


In [ ]:
# Aggregate which items go with which transaction as a list
df_grouped = df.groupby('TransactionNo').agg({
    'Items': list
}).reset_index()
df_grouped

,TransactionNo,Items
0,1,[Bread]
1,2,"[Scandinavian, Scandinavian]"
2,3,"[Hot chocolate, Jam, Cookies]"
3,4,[Muffin]
4,5,"[Coffee, Pastry, Bread]"
...,...,...
9460,9680,[Bread]
9461,9681,"[Truffles, Tea, Spanish Brunch, Christmas common]"
9462,9682,"[Muffin, Tacos/Fajita, Coffee, Tea]"
9463,9683,"[Coffee, Pastry]"


In [ ]:
df_grouped_transactions = df_grouped['Items'].to_list()
df_grouped_transactions

[['Bread'],
 ['Scandinavian', 'Scandinavian'],
 ['Hot chocolate', 'Jam', 'Cookies'],
 ['Muffin'],
 ['Coffee', 'Pastry', 'Bread'],
 ['Medialuna', 'Pastry', 'Muffin'],
 ['Medialuna', 'Pastry', 'Coffee', 'Tea'],
 ['Pastry', 'Bread'],
 ['Bread', 'Muffin'],
 ['Scandinavian', 'Medialuna'],
 ['Bread', 'Medialuna', 'Bread'],
 ['Jam', 'Coffee', 'Tartine', 'Pastry', 'Tea'],
 ['Basket', 'Bread', 'Coffee'],
 ['Bread', 'Medialuna', 'Pastry'],
 ['Mineral water', 'Scandinavian'],
 ['Bread', 'Medialuna', 'Coffee'],
 ['Hot chocolate'],
 ['Farm House'],
 ['Farm House', 'Bread'],
 ['Bread', 'Medialuna'],
 ['Coffee', 'Coffee', 'Medialuna', 'Bread'],
 ['Jam'],
 ['Scandinavian', 'Muffin'],
 ['Bread'],
 ['Scandinavian'],
 ['Fudge'],
 ['Scandinavian'],
 ['Coffee', 'Bread'],
 ['Bread', 'Jam'],
 ['Bread'],
 ['Basket'],
 ['Scandinavian', 'Muffin'],
 ['Coffee'],
 ['Coffee', 'Muffin'],
 ['Muffin', 'Scandinavian'],
 ['Tea', 'Bread'],
 ['Coffee', 'Bread'],
 ['Bread', 'Tea'],
 ['Scandinavian'],
 ['Juice', 'Tartine', 

In [ ]:
# An item must appear at least 10 times in the data to be worth thinking about in terms of support
# Therefore the minimum support will be 10 divided by the amount of transactions
min_support = 10 / len(df_grouped_transactions)
# Min confidence, start at 0.1, so that we get hits on most things
min_confidence = 0.1
# Min lift, start at 1, so that only those items that are as likely or more in the RHS based on the LHS appear
min_lift = 1
# Min and Max length must be 2, to compare one product to another
min_length = 2
max_length = 2
df_grouped_transactions_rules = apriori(transactions = df_grouped_transactions,
                                        min_support=min_support,
                                        min_confidence = min_confidence,
                                        min_lift = min_lift ,
                                        min_length = min_length,
                                        max_length = max_length)
df_grouped_rules_results = list(df_grouped_transactions_rules)
df_grouped_rules_results

[RelationRecord(items=frozenset({'Bread'}), support=0.32720549392498677, ordered_statistics=[OrderedStatistic(items_base=frozenset(), items_add=frozenset({'Bread'}), confidence=0.32720549392498677, lift=1.0)]),
 RelationRecord(items=frozenset({'Cake'}), support=0.10385631273111463, ordered_statistics=[OrderedStatistic(items_base=frozenset(), items_add=frozenset({'Cake'}), confidence=0.10385631273111463, lift=1.0)]),
 RelationRecord(items=frozenset({'Coffee'}), support=0.47839408346539886, ordered_statistics=[OrderedStatistic(items_base=frozenset(), items_add=frozenset({'Coffee'}), confidence=0.47839408346539886, lift=1.0)]),
 RelationRecord(items=frozenset({'Tea'}), support=0.14263074484944532, ordered_statistics=[OrderedStatistic(items_base=frozenset(), items_add=frozenset({'Tea'}), confidence=0.14263074484944532, lift=1.0)]),
 RelationRecord(items=frozenset({'Alfajores', 'Cake'}), support=0.004120443740095087, ordered_statistics=[OrderedStatistic(items_base=frozenset({'Alfajores'}), 

In [ ]:
# The following code works for those items where only a single item was brought
# without breaking the code at lower min_confidences

def inspect(results):
    lhs, rhs, supports, confidences, lifts = [], [], [], [], []

    for r in results:
        try:
            if r[2] and len(r[2]) > 0 and len(r[2][0]) >= 4:
                lhs_item = tuple(r[2][0][0])[0] if r[2][0][0] else None
                rhs_item = tuple(r[2][0][1])[0] if r[2][0][1] else None
                confidence = r[2][0][2]
                lift = r[2][0][3]
            else:
                lhs_item = rhs_item = confidence = lift = None
        except (IndexError, TypeError):
            lhs_item = rhs_item = confidence = lift = None

        lhs.append(lhs_item)
        rhs.append(rhs_item)
        supports.append(r[1])
        confidences.append(confidence)
        lifts.append(lift)

    return list(zip(lhs, rhs, supports, confidences, lifts))

In [ ]:
results_df = pd.DataFrame(inspect(df_grouped_rules_results), columns =
 ['Left Hand Side', 'Right Hand Side', 'Support', 'Confidence', 'Lift'])
# Need to remove those Columns with None on the Left Hand Side, as they only refer to
# when only certain items have been purchased
# and there is no assoication insight there
results_df = results_df[results_df["Left Hand Side"].notna()]
results_df

,Left Hand Side,Right Hand Side,Support,Confidence,Lift
4,Alfajores,Cake,0.004120,0.113372,1.091624
5,Alfajores,Coffee,0.019651,0.540698,1.130235
6,Alfajores,Tea,0.006762,0.186047,1.304393
7,Art Tray,Coffee,0.002747,0.684211,1.430224
8,Art Tray,Tea,0.001162,0.289474,2.029532
...,...,...,...,...,...
98,Spanish Brunch,Tea,0.004649,0.255814,1.793540
99,The Nomad,Tea,0.001796,0.293103,2.054981
100,Tiffin,Tea,0.003487,0.226027,1.584703
101,Toast,Tea,0.006445,0.191824,1.344899


In [ ]:
# Extra Salami and Feta has insane lift
top_10_lift = results_df.sort_values(by="Lift", ascending=False).head(10)
print(top_10_lift)
top_10_support = results_df.sort_values(by="Support", ascending=False).head(10)
print(top_10_support)
top_10_confidence = results_df.sort_values(by="Confidence", ascending=False).head(10)
print(top_10_confidence)

          Left Hand Side Right Hand Side   Support  Confidence       Lift
68  Extra Salami or Feta           Salad  0.001690    0.421053  40.255183
70                 Fudge             Jam  0.002536    0.169014  11.265622
89                 Salad  Spanish Brunch  0.001268    0.121212   6.670190
75        Jammie Dodgers           Juice  0.002113    0.160000   4.149041
80        Spanish Brunch           Juice  0.002747    0.151163   3.919879
85         Mineral water            Soup  0.001902    0.134328   3.900055
64                  Coke        Sandwich  0.005177    0.266304   3.706722
39          Chicken Stew            Soup  0.001585    0.121951   3.540700
88                 Salad            Soup  0.001268    0.121212   3.519241
72     Hearty & Seasonal            Soup  0.001268    0.120000   3.484049
   Left Hand Side Right Hand Side   Support  Confidence      Lift
22           Cake          Coffee  0.054728    0.526958  1.101515
52         Pastry          Coffee  0.047544    0.55214